# Decoder-only Transformer：结构、训练与推理

本文以 **GPT-2 Small**（hidden_size=768，12层，vocab_size=50257）为参照，
逐模块拆解 Decoder-only Transformer 在训练和推理阶段的行为。

## 整体结构

```
输入序列（训练时有标签，推理时只有 prompt）
[token₀, token₁, ..., token₁₀₂₃]   ← seq_len=1024
  ↓
┌──────────────────────────────────────┐
│           Token Embedding            │
│       每个 token → (1, 768)          │
│       整句 → (1024, 768)             │
└──────────────────┬───────────────────┘
                   ↓
┌──────────────────────────────────────┐
│         位置编码（可学习）            │
│       (1024, 768) → (1024, 768)      │
└──────────────────┬───────────────────┘
                   ↓
        ┌──────────┴──────────┐
        │  × 12 层重复         │
        │                     │
        │  ┌───────────────┐  │
        │  │   LayerNorm   │  │
        │  └──────┬────────┘  │
        │         ↓           │
        │  ┌───────────────┐  │
        │  │ Masked Self-  │  │
        │  │   Attention   │  │
        │  │ (Causal Mask) │  │
        │  └──────┬────────┘  │
        │         ↓ + 残差    │
        │  ┌───────────────┐  │
        │  │   LayerNorm   │  │
        │  └──────┬────────┘  │
        │         ↓           │
        │  ┌───────────────┐  │
        │  │  FFN (MLP)    │  │
        │  │    GELU       │  │
        │  └──────┬────────┘  │
        │         ↓ + 残差    │
        └──────────┴──────────┘
                   ↓
┌──────────────────────────────────────┐
│           Final LayerNorm            │
└──────────────────┬───────────────────┘
                   ↓
┌──────────────────────────────────────┐
│    LM Head（与 Embedding 共享权重）   │
│       (1024, 768) → (1024, 50257)    │
└──────────────────┬───────────────────┘
                   ↓
┌──────────────────────────────────────┐
│             Softmax                  │
│   每个位置 → 50257 个词的概率分布     │
└──────────────────────────────────────┘
```

---

### 训练 vs 推理的本质差别

| | 训练 | 推理 |
|---|---|---|
| 并行度 | 全序列并行，一次前向传播算所有位置 | 串行，每次只生成 1 个 token |
| 目标 | 算 Loss，反向传播更新权重 | 权重固定，取概率最大的词 |
| 知道真实答案吗 | 知道，真实下一个 token 就在数据里 | 不知道，要自己生成 |
| Causal Mask | 必须有，防止偷看右边的真实答案 | 天然没有未来，不需要 Mask |

## Token Embedding

一个大小为 `(vocab_size, hidden_size) = (50257, 768)` 的查找表，每个 token id 对应一行向量。

```
token_id=1000 → 查表第 1000 行 → [0.12, -0.34, ..., 0.88]  (768 维)
token_id=2500 → 查表第 2500 行 → [-0.05, 0.67, ..., 0.21]  (768 维)
...
1024 个 token → (1024, 768)
```

加上 batch 维度（实际训练时多个样本并行）：

```
batch_size=8，seq_len=1024

输入张量：      (8, 1024)         ← 每格是一个整数 token id
Embedding 后：  (8, 1024, 768)    ← 每格变成 768 维向量
                 ↑    ↑    ↑
              batch  序列  hidden
```

seq_len=1024 和 hidden=768 是不同的数字，三个维度各自独立。

| | 说明 |
|---|---|
| **训练** | 输入完整序列，token 同时查表，Embedding 权重参与梯度更新 |
| **推理** | 第 1 步只查 prompt 的 token；第 N 步把上一步生成的新 token 追加进来再查表 |

## 位置编码（可学习的绝对位置编码）

Self-Attention 本身是**置换不变**的——打乱词序，点积结果不变。
位置编码把位置信息注入到向量中，让模型区分词序。

GPT-2 用的是**可学习的绝对位置编码**：一张 `(1024, 768)` 的查找表，
位置 0 有一行向量，位置 1 有一行向量，……位置 1023 有一行向量，全部通过训练学出来。

```
token 向量：    (8, 768)     ← 来自 Token Embedding
位置向量：    + (8, 768)     ← 查位置编码表，位置 0~7 各取一行
              = (8, 768)     ← 叠加后输入第一层 Block
```

> 与 LLaMA 用的 RoPE（旋转位置编码）不同：RoPE 在每层 Attention 的 Q/K 上旋转，
> GPT-2 只在输入层叠加一次，更简单，但外推到训练时未见过的更长序列时效果较差。

| | 说明 |
|---|---|
| **训练** | 位置编码表和 Token Embedding 一起参与梯度更新 |
| **推理** | 同训练，按 token 在序列中的绝对位置查表叠加 |

## Masked Self-Attention（Causal Mask）

### Q / K / V 的计算

三个可训练权重矩阵 W_Q / W_K / W_V，shape 均为 `(768, 768)`：

```
Q = X @ W_Q    # (8, 768) @ (768, 768) = (8, 768)
K = X @ W_K    # (8, 768) @ (768, 768) = (8, 768)
V = X @ W_V    # (8, 768) @ (768, 768) = (8, 768)
```

按 12 个 head 拆分（head_dim = 768 / 12 = 64）：

```
Q → (12, 8, 64)
K → (12, 8, 64)
V → (12, 8, 64)
```

### Causal Mask

**训练时**，一次并行算完 8×8 注意力矩阵，Mask 把右上角遮住：

```
        她   把   那   本   书   还   给   他
她  → [ ✓    ✗    ✗    ✗    ✗    ✗    ✗    ✗ ]   ← 只能看自己
把  → [ ✓    ✓    ✗    ✗    ✗    ✗    ✗    ✗ ]
那  → [ ✓    ✓    ✓    ✗    ✗    ✗    ✗    ✗ ]
本  → [ ✓    ✓    ✓    ✓    ✗    ✗    ✗    ✗ ]
书  → [ ✓    ✓    ✓    ✓    ✓    ✗    ✗    ✗ ]
还  → [ ✓    ✓    ✓    ✓    ✓    ✓    ✗    ✗ ]
给  → [ ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✗ ]
他  → [ ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓ ]   ← 能看全部
```

✗ 的位置在 softmax 前被设为 $-\infty$，softmax 之后权重归零。
效果：**用一次前向传播，同时完成了 8 个只看左边的预测任务**。

**推理时**，每步只输入已生成的序列，天然没有未来，不需要 Mask。
KV Cache 缓存历史步骤的 K/V，每步只计算新 token 的 Q 与历史 K/V 做 Attention。

### 单个 head 的完整计算

```
注意力分数 = Q_h @ K_h.T / sqrt(64)
           = (8, 64) @ (64, 8) = (8, 8)     ← 8×8 关系矩阵
           → 加 Causal Mask（右上角 -∞）
           → softmax → 注意力权重 (8, 8)
输出       = 权重 @ V_h = (8, 8) @ (8, 64) = (8, 64)

12 个 head 输出拼接：(8, 12×64) = (8, 768)
最终输出   = 拼接结果 @ W_O = (8, 768)
```

| | 说明 |
|---|---|
| **训练** | 并行算完 8×8 矩阵，Mask 遮右上角；W_Q/W_K/W_V/W_O 参与梯度更新 |
| **推理** | 每步只有最新 token 的 Q 需要计算；历史 K/V 从 KV Cache 读取 |

## FFN（Feed-Forward Network）

Attention 提取 token 间关系之后，FFN 对每个位置单独做非线性变换。
FFN 权重里编码了大量事实性知识。

GPT-2 用标准两层 MLP + GELU 激活：

```
hidden = GELU(X @ W_fc1 + b1)   # (8, 768) @ (768, 3072) = (8, 3072)
out    = hidden @ W_fc2 + b2    # (8, 3072) @ (3072, 768) = (8, 768)
```

intermediate_size = 3072 = 768 × 4，原始 Transformer 论文定的惯例。

> GPT-2 的 FFN 有 bias，而 LLaMA 用的 SwiGLU 无 bias 且有三个矩阵（gate/up/down）。

| | 说明 |
|---|---|
| **训练** | 对每个位置独立做变换，8 个位置完全并行；W_fc1/W_fc2 参与梯度更新 |
| **推理** | 每步只有最新 token 那一行 `(1, 768)` 经过 FFN |

## 残差连接 + LayerNorm（Pre-Norm）

GPT-2 用 **Pre-Norm**：LayerNorm 放在子层**之前**，与原始 Transformer 的 Post-Norm 顺序相反。

```
x'  = x + Attention(LayerNorm(x))
x'' = x' + FFN(LayerNorm(x'))
```

**残差连接**：让梯度直接绕过子层流回早期层，使 12 层深网络能稳定训练。

**Pre-Norm 的好处**：训练更稳定，不容易在深层出现梯度爆炸/消失，
现代大模型（LLaMA 等）也普遍采用 Pre-Norm。

| | 说明 |
|---|---|
| **训练** | LayerNorm 的 $\gamma$/$\beta$ 参与梯度更新；残差确保梯度不消失 |
| **推理** | 行为与训练完全一致，无区别 |

## LM Head + Softmax

经过 12 层 Block 和 Final LayerNorm 后，把每个位置的 768 维向量映射到词表概率分布：

```
logits = hidden @ W_embed.T    # (8, 768) @ (768, 50257) = (8, 50257)
probs  = softmax(logits)        # (8, 50257)，每行加和为 1
```

GPT-2 的 LM Head **共享 Token Embedding 的权重**（转置使用），不额外占参数。

### 训练时：计算 8 个位置的 Loss

真实标签是输入序列**右移一位**：

```
输入:   她(0)  把(1)  那(2)  本(3)  书(4)  还(5)  给(6)  他(7)
标签:   把(1)  那(2)  本(3)  书(4)  还(5)  给(6)  他(7)  <下一个词>
         ↑      ↑      ↑      ↑      ↑      ↑      ↑      ↑
         8 个位置同时预测，8 个 Cross-Entropy Loss 加总平均
```

### 推理时：只看最后一个位置

```
输入 prompt：["小明", "今天"]    → (2, 50257) 的 logits
                                    只取最后一行 (1, 50257)
                                    → argmax 或采样 → "去"

下一步输入：["小明", "今天", "去"]  → 取最后一行 → "学校"
...
```

| | 说明 |
|---|---|
| **训练** | 输出 `(8, 50257)`，每个位置都参与 Loss 计算 |
| **推理** | 只用最后一个位置的输出 `(1, 50257)`，取最大概率得到下一个 token |

## 完整流程对比

### GPT-2 Small 参数一览

| 超参数 | 值 |
|--------|----|
| hidden_size | 768 |
| num_layers | 12 |
| num_heads | 12 |
| head_dim | 64（768/12）|
| intermediate_size | 3072（768×4）|
| vocab_size | 50257 |
| context_len | 1024 |
| 总参数量 | ~117M |

### 训练一个 step

```
从语料取一个 batch（batch_size=8，seq_len=1024）
  ↓
Token Embedding + 位置编码 → (8, 1024, 768)
  ↓
× 12 层：LayerNorm → Masked Self-Attention → 残差
          LayerNorm → FFN → 残差
  ↓
Final LayerNorm → LM Head → (8, 1024, 50257)
  ↓
与右移一位的真实标签算 Cross-Entropy Loss
  ↓
反向传播 → 更新全部 ~117M 个参数
```

### 推理一次生成

```
用户输入 prompt，切成 token 序列
  ↓
第 1 步：前向传播，取最后位置的 logits (1, 50257) → 采样 → token₁
  ↓
第 2 步：把 token₁ 追加到序列，再前向传播 → token₂
  ↓
...循环直到生成 <EOS> 或达到 1024 的上下文长度限制
```

**KV Cache 优化**：推理时历史 token 的 K/V 不变，缓存起来，
每步只计算新 token 的 Q 与历史 K/V 做 Attention，
避免重复计算。代价是显存占用随生成长度线性增长。